# NB_SETUP_POI_ENTITY

Tạo và seed `lh_vv_silver.dbo.ref_dictionary_place_type` (insert-only), gồm `category_group`. `p_apply` mặc định `false`: in kế hoạch rồi thoát, không ghi.

Ghi 5 cạnh vào `slv_poi_entity_non_enrich` và retire 2 cạnh `slv_poi_entity`.


In [ ]:
p_apply = "false"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
# BEGIN_PLACE_TYPE_SEED
import re

SEED_SECTORS = (
    "accommodation",
    "f_n_b",
    "spa_n_wellness",
    "shopping_n_retail",
    "attraction",
    "vehicle_rental",
    "other",
)
SEED_SKIP = (
    "Partner Portal bổ sung",
    "Partner Portal Bổ sung",
    "Fallback",
    "Không có",
)
SEED_ROWS = [
    ('partner', 'Hotel', 'accommodation', 'Hotel', 'Khách sạn', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'hotel', 'accommodation', 'Hotel', 'Khách sạn', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'extended_stay_hotel', 'accommodation', 'Extended Stay Hotel', 'Lưu trú dài hạn', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'inn', 'accommodation', 'Inn', 'Khách sạn nhỏ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Resort', 'accommodation', 'Resort', 'Khu nghỉ dưỡng', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'resort_hotel', 'accommodation', 'Resort', 'Khu nghỉ dưỡng', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('partner', 'Hostel', 'accommodation', 'Hostel', 'Hostel', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'hostel', 'accommodation', 'Hostel', 'Hostel', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('partner', 'Motel', 'accommodation', 'Motel', 'Motel', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'motel', 'accommodation', 'Motel', 'Motel', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('partner', 'Homestay', 'accommodation', 'Homestay', 'Homestay', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'guest_house', 'accommodation', 'Homestay', 'Homestay', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'private_guest_room', 'accommodation', 'Homestay', 'Homestay', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'farmstay', 'accommodation', 'Farmstay', 'Farmstay', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Campground/Glamping', 'accommodation', 'Campground', 'Khu cắm trại', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'campground', 'accommodation', 'Campground', 'Khu cắm trại', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'camping_cabin', 'accommodation', 'Camping Cabin', 'Nhà gỗ cắm trại', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bed_and_breakfast', 'accommodation', 'Bed & Breakfast', 'Bed & Breakfast', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'cottage', 'accommodation', 'Cottage', 'Nhà gỗ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'mobile_home_park', 'accommodation', 'Mobile Home Park', 'Khu nhà lưu động', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'lodging', 'accommodation', 'Accommodation', 'Lưu trú', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'budget_japanese_inn', 'accommodation', 'Inn', 'Khách sạn nhỏ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'japanese_inn', 'accommodation', 'Inn', 'Khách sạn nhỏ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Vietnamese', 'f_n_b', 'Vietnamese restaurant', 'Nhà hàng Việt', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'vietnamese_restaurant', 'f_n_b', 'Vietnamese restaurant', 'Nhà hàng Việt', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Chinese', 'f_n_b', 'Chinese restaurant', 'Nhà hàng Trung', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'chinese_restaurant', 'f_n_b', 'Chinese restaurant', 'Nhà hàng Trung', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Japanese', 'f_n_b', 'Japanese restaurant', 'Nhà hàng Nhật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'japanese_restaurant', 'f_n_b', 'Japanese restaurant', 'Nhà hàng Nhật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Korean', 'f_n_b', 'Korean restaurant', 'Nhà hàng Hàn', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'korean_restaurant', 'f_n_b', 'Korean restaurant', 'Nhà hàng Hàn', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Thai', 'f_n_b', 'Thai restaurant', 'Nhà hàng Thái', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'thai_restaurant', 'f_n_b', 'Thai restaurant', 'Nhà hàng Thái', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'French', 'f_n_b', 'French restaurant', 'Nhà hàng Pháp', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'french_restaurant', 'f_n_b', 'French restaurant', 'Nhà hàng Pháp', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Italian', 'f_n_b', 'Italian restaurant', 'Nhà hàng Ý', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'italian_restaurant', 'f_n_b', 'Italian restaurant', 'Nhà hàng Ý', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'pizza_restaurant', 'f_n_b', 'Italian restaurant', 'Nhà hàng Ý', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'American', 'f_n_b', 'American restaurant', 'Nhà hàng Mỹ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'american_restaurant', 'f_n_b', 'American restaurant', 'Nhà hàng Mỹ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Western', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'european_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'british_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'german_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'spanish_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'greek_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'steak_house', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'portuguese_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'austrian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bavarian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'belgian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'danish_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'dutch_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'scandinavian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'swiss_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'polish_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'romanian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'russian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'ukrainian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'croatian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'czech_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hungarian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'australian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'californian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'tex_mex_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fish_and_chips_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'basque_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'irish_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'soul_food_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'southwestern_us_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'eastern_european_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'International', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'indian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'north_indian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'south_indian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'indonesian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'malaysian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'filipino_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cambodian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'burmese_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'tibetan_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'taiwanese_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'afghani_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'african_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'ethiopian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'moroccan_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'lebanese_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'israeli_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'persian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'turkish_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'middle_eastern_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'mediterranean_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'mexican_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'latin_american_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'south_american_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'brazilian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'argentinian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'colombian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'chilean_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'peruvian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cuban_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'caribbean_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hawaiian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'asian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'asian_fusion_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'mongolian_barbecue_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'pakistani_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bangladeshi_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sri_lankan_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Fusion', 'f_n_b', 'Fusion', 'Ẩm thực Fusion', 'Main Meals', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fusion_restaurant', 'f_n_b', 'Fusion', 'Ẩm thực Fusion', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bistro', 'f_n_b', 'Fusion', 'Ẩm thực Fusion', 'Main Meals', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Bakery & Dessert', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Light Snacks / Cafe', 45, 60, 60, 'all', 'all_year'),
    ('google', 'bakery', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Light Snacks / Cafe', 45, 60, 60, 'all', 'all_year'),
    ('google', 'cake_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'pastry_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'dessert_restaurant', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'dessert_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'donut_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'ice_cream_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'confectionery', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'chocolate_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Bar & Pub/Nightlife', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Shopping & Nightlife', 60, 180, 120, 'all', 'all_year'),
    ('google', 'bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Shopping & Nightlife', 60, 180, 120, 'all', 'all_year'),
    ('google', 'pub', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'night_club', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Shopping & Nightlife', 60, 180, 120, 'all', 'all_year'),
    ('google', 'cocktail_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'wine_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'lounge_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sports_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hookah_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'irish_pub', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'beer_garden', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'karaoke', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Fast Food', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Main Meals', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fast_food_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'meal_takeaway', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Main Meals', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sandwich_shop', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'deli', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'snack_bar', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'burrito_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'taco_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'chicken_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'kebab_shop', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'gyro_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'shawarma_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hamburger_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Vegan Restaurant', 'f_n_b', 'Vegan Restaurant', 'Nhà hàng Chay', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'vegan_restaurant', 'f_n_b', 'Vegan Restaurant', 'Nhà hàng Chay', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'vegetarian_restaurant', 'f_n_b', 'Vegan Restaurant', 'Nhà hàng Chay', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Seafood Restaurant', 'f_n_b', 'Seafood Restaurant', 'Nhà hàng Hải sản', 'Main Meals', 60, 90, 75, 'all', 'all_year'),
    ('google', 'seafood_restaurant', 'f_n_b', 'Seafood Restaurant', 'Nhà hàng Hải sản', 'Main Meals', 60, 90, 75, 'all', 'all_year'),
    ('google', 'oyster_bar_restaurant', 'f_n_b', 'Seafood Restaurant', 'Nhà hàng Hải sản', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Café', 'f_n_b', 'Café', 'Cà phê', 'Light Snacks / Cafe', 45, 60, 60, 'all', 'all_year'),
    ('google', 'coffee_shop', 'f_n_b', 'Café', 'Cà phê', 'Light Snacks / Cafe', 45, 60, 60, 'all', 'all_year'),
    ('google', 'cafe', 'f_n_b', 'Café', 'Cà phê', 'Light Snacks / Cafe', 45, 60, 60, 'all', 'all_year'),
    ('google', 'coffee_roastery', 'f_n_b', 'Café', 'Cà phê', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'coffee_stand', 'f_n_b', 'Café', 'Cà phê', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'tea_house', 'f_n_b', 'Café', 'Cà phê', 'Light Snacks / Cafe', 45, 60, 60, 'all', 'all_year'),
    ('google', 'juice_shop', 'f_n_b', 'Café', 'Cà phê', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hot_pot_restaurant', 'f_n_b', 'Hot Pot', 'Lẩu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'barbecue_restaurant', 'f_n_b', 'Barbecue & Grill', 'Nướng & BBQ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bar_and_grill', 'f_n_b', 'Barbecue & Grill', 'Nướng & BBQ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'korean_barbecue_restaurant', 'f_n_b', 'Barbecue & Grill', 'Nướng & BBQ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'noodle_shop', 'f_n_b', 'Noodle', 'Mì & Món nước', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'ramen_restaurant', 'f_n_b', 'Noodle', 'Mì & Món nước', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'chinese_noodle_restaurant', 'f_n_b', 'Noodle', 'Mì & Món nước', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fine_dining_restaurant', 'f_n_b', 'Fine Dining', 'Fine Dining', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'buffet_restaurant', 'f_n_b', 'Buffet', 'Buffet', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'brunch_restaurant', 'f_n_b', 'Brunch & Breakfast', 'Bữa sáng & Brunch', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'breakfast_restaurant', 'f_n_b', 'Brunch & Breakfast', 'Bữa sáng & Brunch', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'halal_restaurant', 'f_n_b', 'Halal', 'Nhà hàng Halal', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'gastropub', 'f_n_b', 'Gastropub & Brewery', 'Gastropub & Brewery', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'brewpub', 'f_n_b', 'Gastropub & Brewery', 'Gastropub & Brewery', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'brewery', 'f_n_b', 'Gastropub & Brewery', 'Gastropub & Brewery', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'winery', 'f_n_b', 'Gastropub & Brewery', 'Gastropub & Brewery', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'restaurant', 'f_n_b', 'F&B', 'Ẩm thực', 'Main Meals', 60, 90, 75, 'all', 'all_year'),
    ('google', 'food_court', 'f_n_b', 'F&B', 'Ẩm thực', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cafeteria', 'f_n_b', 'F&B', 'Ẩm thực', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'diner', 'f_n_b', 'F&B', 'Ẩm thực', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Day Spa', 'spa_n_wellness', 'Spa', 'Spa', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'spa', 'spa_n_wellness', 'Spa', 'Spa', 'Other', 60, 90, 75, 'spa_wellness', 'all_year'),
    ('google', 'day_spa', 'spa_n_wellness', 'Spa', 'Spa', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'wellness_center', 'spa_n_wellness', 'Spa', 'Spa', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Therapeutic Massage', 'spa_n_wellness', 'Therapeutic Massage', 'Massage trị liệu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'massage', 'spa_n_wellness', 'Therapeutic Massage', 'Massage trị liệu', 'Other', 60, 90, 75, 'spa_wellness', 'all_year'),
    ('partner', 'Clinic-Aesthetic Spa', 'spa_n_wellness', 'Clinic-Aesthetic Spa', 'Viện thẩm mỹ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'skin_care_clinic', 'spa_n_wellness', 'Clinic-Aesthetic Spa', 'Viện thẩm mỹ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'acupuncture_clinic', 'spa_n_wellness', 'Clinic-Aesthetic Spa', 'Viện thẩm mỹ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'ayurvedic_clinic', 'spa_n_wellness', 'Clinic-Aesthetic Spa', 'Viện thẩm mỹ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Blind Massage', 'spa_n_wellness', 'Blind Massage', 'Massage người khiếm thị', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Onsen-Jjimjilbang', 'spa_n_wellness', 'Onsen-Jjimjilbang', 'Tắm Onsen – Xông hơi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hot_spring', 'spa_n_wellness', 'Onsen-Jjimjilbang', 'Tắm Onsen – Xông hơi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sauna', 'spa_n_wellness', 'Onsen-Jjimjilbang', 'Tắm Onsen – Xông hơi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Hair Salon', 'spa_n_wellness', 'Hair Salon', 'Salon', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hair_salon', 'spa_n_wellness', 'Hair Salon', 'Salon', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hair_care', 'spa_n_wellness', 'Hair Salon', 'Salon', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Nail-Lash Salon', 'spa_n_wellness', 'Nail-Lash Salon', 'Tiệm Nail – Mi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'nail_salon', 'spa_n_wellness', 'Nail-Lash Salon', 'Tiệm Nail – Mi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Gym-Fitness', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'gym', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fitness_center', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sports_club', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'recreation_center', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Yoga-Pilates Studio', 'spa_n_wellness', 'Yoga-Pilates Studio', 'Phòng tập Yoga – Pilates', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'yoga_studio', 'spa_n_wellness', 'Yoga-Pilates Studio', 'Phòng tập Yoga – Pilates', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'pilates_studio', 'spa_n_wellness', 'Yoga-Pilates Studio', 'Phòng tập Yoga – Pilates', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Meditation Center', 'spa_n_wellness', 'Meditation Center', 'Trung tâm Thiền', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'meditation_center', 'spa_n_wellness', 'Meditation Center', 'Trung tâm Thiền', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'beauty_salon', 'spa_n_wellness', 'Beauty Salon', 'Spa & Làm đẹp', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hair_removal_service', 'spa_n_wellness', 'Hair Removal Service', 'Dịch vụ triệt lông', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'tanning_studio', 'spa_n_wellness', 'Tanning Studio', 'Dịch vụ nhuộm da', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'barber_shop', 'spa_n_wellness', 'Barbershop', 'Barbershop', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'body_piercing_shop', 'spa_n_wellness', 'Tattoo & Piercing', 'Xăm hình & Xỏ khuyên', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'tattoo_parlor', 'spa_n_wellness', 'Tattoo & Piercing', 'Xăm hình & Xỏ khuyên', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'swimming_pool', 'spa_n_wellness', 'Swimming Pool', 'Bể bơi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Local Specialty', 'shopping_n_retail', 'Local Specialty', 'Đặc sản địa phương', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Souvenirs & Handicrafts', 'shopping_n_retail', 'Souvenirs & Handicrafts', 'Quà lưu niệm & Thủ công mỹ nghệ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'gift_shop', 'shopping_n_retail', 'Souvenirs & Handicrafts', 'Quà lưu niệm & Thủ công mỹ nghệ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'antique_shop', 'shopping_n_retail', 'Souvenirs & Handicrafts', 'Quà lưu niệm & Thủ công mỹ nghệ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Fashion & Accessories', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'clothing_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'shoe_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fashion_accessories_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hat_shop', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sunglasses_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'swimwear_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'leather_goods_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bag_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Jewelry & Gems', 'shopping_n_retail', 'Jewelry & Gems', 'Trang sức & Đá quý', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'jewelry_store', 'shopping_n_retail', 'Jewelry & Gems', 'Trang sức & Đá quý', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Cosmetics & Beauty', 'shopping_n_retail', 'Cosmetics & Beauty', 'Mỹ phẩm & Làm đẹp', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cosmetics_store', 'shopping_n_retail', 'Cosmetics & Beauty', 'Mỹ phẩm & Làm đẹp', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'perfume_store', 'shopping_n_retail', 'Cosmetics & Beauty', 'Mỹ phẩm & Làm đẹp', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'grocery_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'food_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'liquor_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'home_goods_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'furniture_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'home_improvement_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hardware_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'book_store', 'shopping_n_retail', 'Books & Stationery', 'Sách – Văn phòng phẩm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'stationery_store', 'shopping_n_retail', 'Books & Stationery', 'Sách – Văn phòng phẩm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'electronics_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cell_phone_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'shopping_mall', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị', 'Shopping & Nightlife', 60, 180, 120, 'shopping_nightlife', 'all_year'),
    ('google', 'supermarket', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'department_store', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'discount_store', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'wholesaler', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Convenience Store', 'shopping_n_retail', 'Convenience Store', 'Cửa hàng tiện lợi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'convenience_store', 'shopping_n_retail', 'Convenience Store', 'Cửa hàng tiện lợi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Local/Night Market', 'shopping_n_retail', 'Local/ Night Market', 'Chợ truyền thống / Chợ đêm', 'Shopping & Nightlife', 60, 180, 120, 'all', 'all_year'),
    ('google', 'market', 'shopping_n_retail', 'Local/ Night Market', 'Chợ truyền thống / Chợ đêm', 'Shopping & Nightlife', 90, 150, 120, 'shopping_nightlife', 'all_year'),
    ('partner', 'Sporting Goods', 'shopping_n_retail', 'Sporting Goods', 'Dụng cụ thể thao', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sporting_goods_store', 'shopping_n_retail', 'Sporting Goods', 'Dụng cụ thể thao', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bicycle_store', 'shopping_n_retail', 'Sporting Goods', 'Dụng cụ thể thao', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'toy_store', 'shopping_n_retail', 'Toy Store', 'Cửa hàng đồ chơi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'baby_goods_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'luggage_store', 'shopping_n_retail', 'Luggage Store', 'Vali & Phụ kiện du lịch', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'second_hand_shop', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'florist', 'shopping_n_retail', 'Florist', 'Cửa hàng hoa', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Home & Decor', 'shopping_n_retail', 'Home & Decor', 'Đồ gia dụng - Gốm sứ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Theme Park', 'attraction', 'Theme Park', 'Công viên giải trí', 'Heavy Entertainment', 180, 360, 270, 'all', 'all_year'),
    ('google', 'amusement_park', 'attraction', 'Theme Park', 'Công viên giải trí', 'Heavy Entertainment', 180, 360, 270, 'outdoor_entertainment', 'Month 11–4'),
    ('google', 'amusement_center', 'attraction', 'Theme Park', 'Công viên giải trí', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fairground', 'attraction', 'Theme Park', 'Công viên giải trí', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Water Park', 'attraction', 'Water Park', 'Công viên nước', 'Heavy Entertainment', 180, 360, 270, 'all', 'all_year'),
    ('google', 'water_park', 'attraction', 'Water Park', 'Công viên nước', 'Heavy Entertainment', 180, 360, 270, 'outdoor_entertainment', 'Month 11–4'),
    ('partner', 'Museum', 'attraction', 'Museum', 'Bảo tàng', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'all', 'all_year'),
    ('google', 'museum', 'attraction', 'Museum', 'Bảo tàng', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'indoor_culture', 'all_year'),
    ('google', 'planetarium', 'attraction', 'Museum', 'Bảo tàng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Historical Site', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'all', 'all_year'),
    ('google', 'historical_landmark', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'all', 'all_year'),
    ('google', 'archaeological_site', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'ruins', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'castle', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'palace', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'monument', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sculpture', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'lighthouse', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Zoo-Aquarium', 'attraction', 'Zoo-Aquarium', 'Vườn thú – Thủy cung', 'Heavy Entertainment', 180, 360, 270, 'all', 'all_year'),
    ('google', 'zoo', 'attraction', 'Zoo-Aquarium', 'Vườn thú – Thủy cung', 'Heavy Entertainment', 180, 360, 270, 'all', 'all_year'),
    ('google', 'aquarium', 'attraction', 'Zoo-Aquarium', 'Vườn thú – Thủy cung', 'Heavy Entertainment', 180, 360, 270, 'all', 'all_year'),
    ('google', 'wildlife_park', 'attraction', 'Zoo-Aquarium', 'Vườn thú – Thủy cung', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Nature Reserve', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên', 'Heavy Entertainment', 180, 360, 270, 'all', 'all_year'),
    ('google', 'nature_reserve', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'national_park', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên', 'Heavy Entertainment', 180, 360, 270, 'forest_hiking', 'Month 11–3'),
    ('google', 'hiking_area', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên', 'Heavy Entertainment', 180, 360, 270, 'forest_hiking', 'Month 11–3'),
    ('google', 'wildlife_refuge', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Architecture-Landmark', 'attraction', 'Architecture-Landmark', 'Công trình kiến trúc', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cultural_landmark', 'attraction', 'Architecture-Landmark', 'Công trình kiến trúc', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Art Show-Performance', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'concert_hall', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'performing_arts_theater', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'opera_house', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'comedy_club', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'amphitheatre', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Traditional Village', 'attraction', 'Traditional Village', 'Làng nghề', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Religious Site', 'attraction', 'Religious Site', 'Điểm đến tâm linh', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'all', 'all_year'),
    ('google', 'church', 'attraction', 'Religious Site', 'Điểm đến tâm linh', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'indoor_culture', 'all_year'),
    ('google', 'hindu_temple', 'attraction', 'Religious Site', 'Điểm đến tâm linh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'mosque', 'attraction', 'Religious Site', 'Điểm đến tâm linh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'synagogue', 'attraction', 'Religious Site', 'Điểm đến tâm linh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'buddhist_temple', 'attraction', 'Religious Site', 'Điểm đến tâm linh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'place_of_worship', 'attraction', 'Religious Site', 'Điểm đến tâm linh', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'indoor_culture', 'all_year'),
    ('partner', 'Casino', 'attraction', 'Casino', 'Sòng bạc', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'casino', 'attraction', 'Casino', 'Sòng bạc', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Art Gallery', 'attraction', 'Art Gallery', 'Phòng tranh/ Xưởng nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'art_gallery', 'attraction', 'Art Gallery', 'Phòng tranh/ Xưởng nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cultural_center', 'attraction', 'Cultural Center', 'Trung tâm văn hóa', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'community_center', 'attraction', 'Cultural Center', 'Trung tâm văn hóa', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'escape_room', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'video_arcade', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bowling_alley', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'miniature_golf_course', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'go_kart_track', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'beach', 'attraction', 'Beach', 'Biển', 'Light Sightseeing / Culture & Nature', 120, 240, 180, 'beach_coast', 'Month 12–4'),
    ('google', 'park', 'attraction', 'Park', 'Công viên', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'all', 'all_year'),
    ('google', 'viewpoint', 'attraction', 'Viewpoint', 'Điểm ngắm cảnh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'observation_deck', 'attraction', 'Observation Deck', 'Đài quan sát', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'waterfall', 'attraction', 'Waterfall', 'Thác nước', 'Other', 60, 90, 75, 'waterfall_green_season', 'Month 6–10'),
    ('google', 'cave', 'attraction', 'Cave', 'Hang động', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'botanical_garden', 'attraction', 'Botanical Garden', 'Vườn bách thảo', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'stadium', 'attraction', 'Stadium', 'Sân vận động', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Motorbike Rental', 'vehicle_rental', 'Motorbike Rental', 'Cho thuê xe máy', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'scooter_rental', 'vehicle_rental', 'Motorbike Rental', 'Cho thuê xe máy', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Self-drive Car Rental', 'vehicle_rental', 'Self-drive Car Rental', 'Cho thuê ô tô tự lái', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'car_rental', 'vehicle_rental', 'Self-drive Car Rental', 'Cho thuê ô tô tự lái', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Car with Driver', 'vehicle_rental', None, None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Bicycle-EV Rental', 'vehicle_rental', 'Bicycle-EV Rental', 'Cho thuê xe đạp – Xe điện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bicycle_rental', 'vehicle_rental', 'Bicycle-EV Rental', 'Cho thuê xe đạp – Xe điện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Campervan Rental', 'vehicle_rental', None, 'Cho thuê xe nhà di động', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'boat_rental', 'vehicle_rental', 'Boat Rental', 'Cho thuê thuyền', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'airport', 'other', 'airport', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'international_airport', 'other', 'airport', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bus_station', 'other', 'bus_station', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'train_station', 'other', 'train_station', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'subway_station', 'other', 'metro_station', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'light_rail_station', 'other', 'metro_station', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'ferry_terminal', 'other', 'ferry_terminal', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'transit_station', 'other', 'transit_station', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'natural_feature', 'other', 'natural_feature', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'town_square', 'other', 'town_square', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'city_hall', 'other', 'government_offices', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'local_government_office', 'other', 'government_offices', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'embassy', 'other', 'embassy', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'police', 'other', 'police_and_fire_station', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fire_station', 'other', 'police_and_fire_station', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'post_office', 'other', 'post_office', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'library', 'other', 'public_library', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hospital', 'other', 'hospital', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'drugstore', 'other', 'drugstore', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'university', 'other', 'academic_building', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'school', 'other', 'academic_building', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cable_car', 'other', 'other', 'other', 'Other', 180, 360, 270, 'all', 'all_year'),
    ('google', 'safari', 'other', 'other', 'other', 'Other', 180, 360, 270, 'all', 'all_year'),
    ('google', 'theme', 'other', 'other', 'other', 'Other', 180, 360, 270, 'all', 'all_year'),
    ('google', 'vinwonders', 'other', 'other', 'other', 'Other', 180, 360, 270, 'all', 'all_year'),
    ('google', 'night_market', 'other', 'other', 'other', 'Shopping & Nightlife', 90, 150, 120, 'shopping_nightlife', 'all_year'),
    ('google', 'pagoda', 'other', 'other', 'other', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'indoor_culture', 'all_year'),
    ('google', 'temple', 'other', 'other', 'other', 'Other', 60, 120, 90, 'indoor_culture', 'all_year'),
    ('google', 'prison', 'other', 'other', 'other', 'Other', 60, 120, 90, 'indoor_culture', 'all_year'),
    ('google', 'snorkel', 'other', 'other', 'other', 'Other', 120, 240, 180, 'snorkeling_diving', 'Month 12–5'),
    ('google', 'island', 'other', 'other', 'other', 'Other', 120, 240, 180, 'island_boat', 'Month 11–4'),
    ('google', 'boat', 'other', 'other', 'other', 'Other', 120, 240, 180, 'island_boat', 'Month 11–4'),
    ('google', 'snorkeling', 'other', 'other', 'other', 'Other', 60, 90, 75, 'snorkeling_diving', 'Month 12–5'),
    ('google', 'diving', 'other', 'other', 'other', 'Other', 60, 90, 75, 'snorkeling_diving', 'Month 12–5'),
    ('google', 'scuba', 'other', 'other', 'other', 'Other', 60, 90, 75, 'snorkeling_diving', 'Month 12–5'),
    ('google', 'canoe', 'other', 'other', 'other', 'Other', 60, 90, 75, 'island_boat', 'Month 11–4'),
    ('google', 'ferry', 'other', 'other', 'other', 'Other', 60, 90, 75, 'island_boat', 'Month 11–4'),
    ('google', 'sailing', 'other', 'other', 'other', 'Other', 60, 90, 75, 'island_boat', 'Month 11–4'),
    ('google', 'sunset', 'other', 'other', 'other', 'Other', 60, 90, 75, 'beach_coast', 'Month 12–4'),
    ('google', 'coast', 'other', 'other', 'other', 'Other', 60, 90, 75, 'beach_coast', 'Month 12–4'),
    ('google', 'coastal', 'other', 'other', 'other', 'Other', 60, 90, 75, 'beach_coast', 'Month 12–4'),
    ('google', 'stream', 'other', 'other', 'other', 'Other', 60, 90, 75, 'waterfall_green_season', 'Month 6–10'),
    ('google', 'suối', 'other', 'other', 'other', 'Other', 60, 90, 75, 'waterfall_green_season', 'Month 6–10'),
    ('google', 'thác', 'other', 'other', 'other', 'Other', 60, 90, 75, 'waterfall_green_season', 'Month 6–10'),
    ('google', 'forest', 'other', 'other', 'other', 'Other', 60, 90, 75, 'forest_hiking', 'Month 11–3'),
    ('google', 'hiking', 'other', 'other', 'other', 'Other', 60, 90, 75, 'forest_hiking', 'Month 11–3'),
    ('google', 'trekking', 'other', 'other', 'other', 'Other', 60, 90, 75, 'forest_hiking', 'Month 11–3'),
    ('google', 'trail', 'other', 'other', 'other', 'Other', 60, 90, 75, 'forest_hiking', 'Month 11–3'),
    ('google', 'shopping', 'other', 'other', 'other', 'Other', 60, 90, 75, 'shopping_nightlife', 'all_year'),
    ('google', 'mall', 'other', 'other', 'other', 'Other', 60, 90, 75, 'shopping_nightlife', 'all_year'),
    ('google', 'wellness', 'other', 'other', 'other', 'Other', 60, 90, 75, 'spa_wellness', 'all_year'),
    ('google', 'theme_park', 'other', 'other', 'other', 'Heavy Entertainment', 180, 360, 270, 'outdoor_entertainment', 'Month 11–4'),
]
CATEGORY_OTHER = "Other"
# Nhãn mục 1.8. Chỉ để kiểm tra; giá trị ghi sẵn trong từng dòng SEED_ROWS.
CATEGORY_GROUP_VALUES = (
    "Lodging", "Main Meals", "Light Snacks / Cafe", "Heavy Entertainment",
    "Light Sightseeing / Culture & Nature", "Shopping & Nightlife", CATEGORY_OTHER,
)
# Mục 1.9.2: candidate_best_time_code -> canonical_candidate. Chỉ để kiểm tra.
BEST_TIME_CANONICAL = {
    "snorkeling_diving": "Month 12–5", "island_boat": "Month 11–4", "beach_coast": "Month 12–4",
    "waterfall_green_season": "Month 6–10", "forest_hiking": "Month 11–3",
    "outdoor_entertainment": "Month 11–4", "indoor_culture": "all_year",
    "shopping_nightlife": "all_year", "spa_wellness": "all_year", "all": "all_year",
}
TOKEN_KEEP = ("suối", "thác")  # D-04: giữ đúng chữ


def validate_seed(rows):
    """Khóa trùng, source, sector, placeholder, token google, category_group, ước tính 1.9."""
    seen = set()
    token_re = re.compile(r"^[a-z0-9_]+$")
    allowed = set(SEED_SECTORS)
    skip = set(SEED_SKIP)
    for row in rows:
        if len(row) != 11:
            raise ValueError(f"{row[:2]}: {len(row)} trường, cần 11")
        source, place_type, sector, _cat, _cat_vi, group, low, high, avg, code, canonical = row
        key = (source, place_type)
        if key in seen:
            raise ValueError(f"duplicate key {key}")
        seen.add(key)
        if source not in {"google", "partner"}:
            raise ValueError(f"source {source!r}")
        if sector not in allowed:
            raise ValueError(f"sector {sector!r}")
        if place_type in skip:
            raise ValueError(f"place_type {place_type!r}")
        if source == "google" and place_type not in TOKEN_KEEP and token_re.fullmatch(place_type) is None:
            raise ValueError(f"bad token {place_type!r}")
        if group not in CATEGORY_GROUP_VALUES:
            raise ValueError(f"{key}: category_group {group!r}")
        if not all(type(v) is int for v in (low, high, avg)) or not low <= avg <= high:
            raise ValueError(f"{key}: range {low}-{high} avg {avg}")
        if BEST_TIME_CANONICAL.get(code) != canonical:
            raise ValueError(f"{key}: best time {code!r} -> {canonical!r}")
# END_PLACE_TYPE_SEED


In [ ]:
import json
from collections import Counter

REF_FQ = "lh_vv_silver.dbo.ref_dictionary_place_type"
SEED_SCHEMA = (
    "source_name string, place_type string, business_sector string, "
    "business_category string, business_category_vi string, category_group string, "
    "range_low_min int, range_high_min int, avg_range_min int, "
    "candidate_best_time_code string, canonical_candidate string"
)
apply = to_bool(p_apply)
validate_seed(SEED_ROWS)
counts = Counter(row[0] for row in SEED_ROWS)
seed_df = spark.createDataFrame(SEED_ROWS, SEED_SCHEMA)
seed_df.createOrReplaceTempView("seed_place_type")
if table_exists(REF_FQ):
    missing_keys = spark.sql(f"""
        SELECT count(*) AS n
        FROM seed_place_type s
        LEFT ANTI JOIN {REF_FQ} t
          ON t.source_name = s.source_name AND t.place_type = s.place_type
    """).first()["n"]
else:
    missing_keys = 0
planned_edges = [
    "slv_3p_poi", "slv_3p_poi_address", "ref_dictionary_place_type",
    "slv_pn_business_services", "slv_pn_business_service_i18ns",
]
planned_retire = ["slv_3p_poi", "ref_dictionary_place_type"]
ESTIMATE_COLS = ("category_group", "range_low_min", "range_high_min", "avg_range_min",
                 "candidate_best_time_code", "canonical_candidate")
unseeded = 0
if table_exists(REF_FQ):
    have = {field.name.lower() for field in spark.table(REF_FQ).schema.fields}
    if set(ESTIMATE_COLS) <= have:
        cond = " OR ".join(f"{col} IS NULL" for col in ESTIMATE_COLS)
        unseeded = spark.sql(f"SELECT count(*) AS n FROM {REF_FQ} WHERE {cond}").first()["n"]
    else:
        unseeded = -1
print(f"unseeded_rows={unseeded}")
print(f"seed google={counts['google']} partner={counts['partner']} missing_keys={missing_keys}")
print(f"planned edges: {len(planned_edges)} UPSERT -> slv_poi_entity_non_enrich; retire {len(planned_retire)} -> slv_poi_entity")
if not apply:
    notebookutils.notebook.exit(json.dumps({
        "status": "PLAN",
        "p_apply": False,
        "google": counts["google"],
        "partner": counts["partner"],
        "missing_keys": int(missing_keys),
        "unseeded_rows": int(unseeded),
        "planned_edges": planned_edges,
        "planned_retire": planned_retire,
    }))

if unseeded:
    raise ConfigError(f"{REF_FQ}: {unseeded} dòng thiếu cột/giá trị 1.8–1.9 (-1 = schema cũ). "
                      "Chạy runbook 20261020-dictionary-seed-from-mapping: xoá và init lại")


In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lh_vv_silver.dbo.ref_dictionary_place_type (
    source_name          STRING NOT NULL,
    place_type           STRING NOT NULL,
    business_sector      STRING,
    business_category    STRING,
    business_category_vi STRING,
    is_active            BOOLEAN,
    updated_at           TIMESTAMP,
    category_group       STRING,
    range_low_min        INT,
    range_high_min       INT,
    avg_range_min        INT,
    candidate_best_time_code STRING,
    canonical_candidate  STRING
) USING DELTA
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true', 'delta.autoOptimize.autoCompact' = 'true')
""")
EXPECTED = [
    ("source_name", "string", False),
    ("place_type", "string", False),
    ("business_sector", "string", True),
    ("business_category", "string", True),
    ("business_category_vi", "string", True),
    ("is_active", "boolean", True),
    ("updated_at", "timestamp", True),
    ("category_group", "string", True),
    ("range_low_min", "int", True),
    ("range_high_min", "int", True),
    ("avg_range_min", "int", True),
    ("candidate_best_time_code", "string", True),
    ("canonical_candidate", "string", True),
]
described = spark.table(REF_FQ).schema.fields
got = [(field.name.lower(), field.dataType.simpleString(), field.nullable) for field in described]
if got != EXPECTED:
    raise ConfigError(f"{REF_FQ} lệch cột hoặc nullability: có {got}, cần {EXPECTED}. Thiếu cột 1.9: chạy runbook 20261020-dictionary-seed-from-mapping (xoá và init lại)")


In [ ]:
def _seed_merge():
    spark.sql("""
        MERGE INTO lh_vv_silver.dbo.ref_dictionary_place_type AS t
        USING seed_place_type AS s
        ON t.source_name = s.source_name AND t.place_type = s.place_type
        WHEN NOT MATCHED THEN INSERT
            (source_name, place_type, business_sector, business_category, business_category_vi, is_active, updated_at, category_group, range_low_min, range_high_min, avg_range_min, candidate_best_time_code, canonical_candidate)
        VALUES (s.source_name, s.place_type, s.business_sector, s.business_category, s.business_category_vi,
                true, current_timestamp(), s.category_group, s.range_low_min, s.range_high_min, s.avg_range_min, s.candidate_best_time_code, s.canonical_candidate)
    """)


missing_before = spark.sql("""
    SELECT count(*) AS n
    FROM seed_place_type s
    LEFT ANTI JOIN lh_vv_silver.dbo.ref_dictionary_place_type t
      ON t.source_name = s.source_name AND t.place_type = s.place_type
""").first()["n"]
with_retry(_seed_merge, "seed ref_dictionary_place_type")
INSERTED = int(missing_before)

dups = spark.sql("""
    SELECT count(*) AS n FROM (
        SELECT source_name, place_type
        FROM lh_vv_silver.dbo.ref_dictionary_place_type
        GROUP BY source_name, place_type
        HAVING count(*) > 1
    )
""").first()["n"]
if dups:
    raise ConfigError(f"ref_dictionary_place_type: {dups} khoá (source_name, place_type) trùng")

missing_after = spark.sql("""
    SELECT count(*) AS n
    FROM seed_place_type s
    LEFT ANTI JOIN lh_vv_silver.dbo.ref_dictionary_place_type t
      ON t.source_name = s.source_name AND t.place_type = s.place_type
""").first()["n"]
if missing_after:
    raise ConfigError(f"ref_dictionary_place_type: thiếu {missing_after} khoá seed")

DRIFT = int(spark.sql("""
    SELECT count(*) AS n
    FROM seed_place_type s
    JOIN lh_vv_silver.dbo.ref_dictionary_place_type t
      ON t.source_name = s.source_name AND t.place_type = s.place_type
    WHERE NOT (
        t.business_sector <=> s.business_sector
        AND t.business_category <=> s.business_category
        AND t.business_category_vi <=> s.business_category_vi
        AND t.category_group <=> s.category_group
        AND t.range_low_min <=> s.range_low_min
        AND t.range_high_min <=> s.range_high_min
        AND t.avg_range_min <=> s.avg_range_min
        AND t.candidate_best_time_code <=> s.candidate_best_time_code
        AND t.canonical_candidate <=> s.canonical_candidate
        AND t.is_active = true
    )
""").first()["n"])
print(f"inserted={INSERTED} drift={DRIFT}")


In [ ]:
import uuid

# BEGIN_EDGE_PRECHECK
def setup_edge_issues(rows, upserts, retires):
    """Cạnh active trên lh_vv_silver.dbo mà MERGE không ghi thì chặn. Đúng bộ RETIRE thì cho qua.

    rows: pl_name, src_schema, src_tbl, trg_schema, trg_tbl, is_active.
    upserts, retires: tập (src_tbl, trg_tbl) của MERGE này, schema lh_vv_silver.dbo, pl SLV_1H.
    Schema khác không tính. Output: list thông báo, rỗng nghĩa là được MERGE.
    """
    silver = "lh_vv_silver.dbo"
    flow = "PL_VV_TRANS_SLV_TO_SLV_1H"
    targets = {trg for _src, trg in upserts}
    extra, stale_entity, stale_amenity = [], 0, 0
    for row in rows:
        if int(row.get("is_active") or 0) != 1:
            continue
        if row.get("src_schema") != silver or row.get("trg_schema") != silver:
            continue
        key = (row["src_tbl"], row["trg_tbl"])
        planned_retire = row.get("pl_name") == flow and key in retires
        if row["trg_tbl"] in targets and key not in upserts and not planned_retire:
            extra.append(f"{row['trg_tbl']}/{row['pl_name']}/{row['src_tbl']}")
        if row["trg_tbl"] == "slv_poi_entity" and not planned_retire:
            stale_entity += 1
        if key == ("slv_poi_entity", "slv_poi_amenity") and row.get("pl_name") != flow:
            stale_amenity += 1
    issues = []
    if extra:
        issues.append("còn cạnh active ngoài cạnh chuẩn: " + ", ".join(extra))
    if stale_entity:
        issues.append(f"slv_poi_entity: còn {stale_entity} cạnh active")
    if stale_amenity:
        issues.append("slv_poi_amenity: còn cạnh active slv_poi_entity")
    return issues
# END_EDGE_PRECHECK


def _write_edges():
    """Ghi cạnh entity và amenity trong một MERGE. Kiểm tra dưới lock, trước khi commit."""
    cfg = "lh_vv_ctrl.dbo.ctrl_mng_pipeline_config"
    has_nb = "nb_name" in table_columns(cfg)
    nb_match = " AND t.nb_name <=> CAST(NULL AS STRING)" if has_nb else ""
    nb_set = ", nb_name = NULL" if has_nb else ""
    nb_col = ", nb_name" if has_nb else ""
    nb_val = ", CAST(NULL AS STRING)" if has_nb else ""
    edge_rows = """
            (1, 'lh_vv_silver.dbo', 'slv_3p_poi', 'lh_vv_silver.dbo', 'slv_poi_entity_non_enrich', 'UPSERT'),
            (2, 'lh_vv_silver.dbo', 'slv_3p_poi_address', 'lh_vv_silver.dbo', 'slv_poi_entity_non_enrich', 'UPSERT'),
            (3, 'lh_vv_silver.dbo', 'ref_dictionary_place_type', 'lh_vv_silver.dbo', 'slv_poi_entity_non_enrich', 'UPSERT'),
            (4, 'lh_vv_silver.dbo', 'slv_pn_business_services', 'lh_vv_silver.dbo', 'slv_poi_entity_non_enrich', 'UPSERT'),
            (5, 'lh_vv_silver.dbo', 'slv_pn_business_service_i18ns', 'lh_vv_silver.dbo', 'slv_poi_entity_non_enrich', 'UPSERT'),
            (0, 'lh_vv_silver.dbo', 'slv_3p_poi', 'lh_vv_silver.dbo', 'slv_poi_entity', 'RETIRE'),
            (0, 'lh_vv_silver.dbo', 'ref_dictionary_place_type', 'lh_vv_silver.dbo', 'slv_poi_entity', 'RETIRE'),
            (6, 'lh_vv_silver.dbo', 'slv_3p_poi_amenity', 'lh_vv_silver.dbo', 'slv_poi_amenity', 'UPSERT'),
            (7, 'lh_vv_silver.dbo', 'slv_poi_entity_non_enrich', 'lh_vv_silver.dbo', 'slv_poi_amenity', 'UPSERT'),
            (8, 'lh_vv_silver.dbo', 'ref_dictionary_amenities', 'lh_vv_silver.dbo', 'slv_poi_amenity', 'UPSERT'),
            (0, 'lh_vv_silver.dbo', 'slv_poi_entity', 'lh_vv_silver.dbo', 'slv_poi_amenity', 'RETIRE')"""
    dup = spark.sql(f"""
        SELECT trg_tbl, src_tbl, count(*) AS n
        FROM {cfg}
        WHERE pl_name = 'PL_VV_TRANS_SLV_TO_SLV_1H'
          AND src_schema = 'lh_vv_silver.dbo'
          AND trg_schema = 'lh_vv_silver.dbo'
          AND (
            (trg_tbl = 'slv_poi_entity_non_enrich' AND src_tbl IN (
                'slv_3p_poi', 'slv_3p_poi_address', 'ref_dictionary_place_type',
                'slv_pn_business_services', 'slv_pn_business_service_i18ns'))
            OR (trg_tbl = 'slv_poi_entity' AND src_tbl IN ('slv_3p_poi', 'ref_dictionary_place_type'))
            OR (trg_tbl = 'slv_poi_amenity' AND src_tbl IN (
                'slv_3p_poi_amenity', 'slv_poi_entity_non_enrich', 'ref_dictionary_amenities', 'slv_poi_entity'))
          )
        GROUP BY src_schema, src_tbl, trg_schema, trg_tbl
        HAVING count(*) > 1
    """).collect()
    if dup:
        names = ", ".join(f"{row.trg_tbl}/{row.src_tbl}={row.n}" for row in dup)
        raise ConfigError(f"slv_poi_entity_non_enrich: trùng cạnh, không ghi đè; cạnh active không đúng: {names}")
    owners = spark.sql(f"""
        SELECT DISTINCT trg_tbl, pl_name
        FROM {cfg}
        WHERE trg_schema = 'lh_vv_silver.dbo'
          AND trg_tbl IN ('slv_poi_entity_non_enrich', 'slv_poi_amenity')
          AND is_active = 1
          AND pl_name <> 'PL_VV_TRANS_SLV_TO_SLV_1H'
    """).collect()
    if owners:
        names = ", ".join(f"{row.trg_tbl}/{row.pl_name}" for row in owners)
        raise ConfigError(f"slv_poi_entity_non_enrich đã thuộc pl_name khác: {names}")
    watched = spark.sql(f"""
        SELECT pl_name, src_schema, src_tbl, trg_schema, trg_tbl, is_active
        FROM {cfg}
        WHERE src_schema = 'lh_vv_silver.dbo' AND trg_schema = 'lh_vv_silver.dbo'
          AND (
            trg_tbl IN ('slv_poi_entity_non_enrich', 'slv_poi_amenity', 'slv_poi_entity')
            OR src_tbl = 'slv_poi_entity'
          )
    """).collect()
    entity_upserts = {
        ("slv_3p_poi", "slv_poi_entity_non_enrich"),
        ("slv_3p_poi_address", "slv_poi_entity_non_enrich"),
        ("ref_dictionary_place_type", "slv_poi_entity_non_enrich"),
        ("slv_pn_business_services", "slv_poi_entity_non_enrich"),
        ("slv_pn_business_service_i18ns", "slv_poi_entity_non_enrich"),
        ("slv_3p_poi_amenity", "slv_poi_amenity"),
        ("slv_poi_entity_non_enrich", "slv_poi_amenity"),
        ("ref_dictionary_amenities", "slv_poi_amenity"),
    }
    entity_retires = {
        ("slv_3p_poi", "slv_poi_entity"),
        ("ref_dictionary_place_type", "slv_poi_entity"),
        ("slv_poi_entity", "slv_poi_amenity"),
    }
    issues = setup_edge_issues([row.asDict() for row in watched], entity_upserts, entity_retires)
    if issues:
        raise ConfigError("; ".join(issues))

    missing = spark.sql(f"""
        SELECT count(*) AS n
        FROM VALUES {edge_rows} AS t(seq, src_schema, src_tbl, trg_schema, trg_tbl, action)
        WHERE t.action = 'UPSERT' AND NOT EXISTS (
            SELECT 1 FROM {cfg} c
            WHERE c.pl_name = 'PL_VV_TRANS_SLV_TO_SLV_1H'
              AND c.src_schema = t.src_schema AND c.src_tbl = t.src_tbl
              AND c.trg_schema = t.trg_schema AND c.trg_tbl = t.trg_tbl)
    """).first()["n"]
    retired = spark.sql(f"""
        SELECT count(*) AS n FROM {cfg}
        WHERE pl_name = 'PL_VV_TRANS_SLV_TO_SLV_1H'
          AND src_schema = 'lh_vv_silver.dbo'
          AND trg_schema = 'lh_vv_silver.dbo'
          AND (
            (trg_tbl = 'slv_poi_entity' AND src_tbl = 'slv_3p_poi' AND NOT (is_active <=> 0))
            OR (trg_tbl = 'slv_poi_entity' AND src_tbl = 'ref_dictionary_place_type' AND NOT (is_active <=> 0))
            OR (trg_tbl = 'slv_poi_amenity' AND src_tbl = 'slv_poi_entity' AND NOT (is_active <=> 0))
          )
    """).first()["n"]
    spark.sql(f"""
        MERGE INTO {cfg} AS t
        USING (
            SELECT seq, src_schema, src_tbl, trg_schema, trg_tbl, action
            FROM VALUES {edge_rows} AS v(seq, src_schema, src_tbl, trg_schema, trg_tbl, action)
        ) AS s
        ON t.pl_name = 'PL_VV_TRANS_SLV_TO_SLV_1H'
           AND t.src_schema = s.src_schema
           AND t.src_tbl = s.src_tbl
           AND t.trg_schema = s.trg_schema
           AND t.trg_tbl = s.trg_tbl
        WHEN MATCHED AND s.action = 'RETIRE' AND NOT (t.is_active <=> 0) THEN UPDATE SET
            is_active = 0
        WHEN MATCHED AND s.action = 'UPSERT' AND NOT (
            t.is_active <=> 1 AND t.priority <=> 1 AND t.load_mode <=> 'RECOMPUTE'
            AND t.src_schema <=> 'lh_vv_silver.dbo' AND t.trg_schema <=> 'lh_vv_silver.dbo'
            AND t.align_path <=> CAST(NULL AS STRING) AND t.dedup_order <=> CAST(NULL AS STRING){nb_match}
        ) THEN UPDATE SET
            src_schema = s.src_schema,
            trg_schema = s.trg_schema,
            is_active = 1,
            priority = 1,
            load_mode = 'RECOMPUTE',
            align_path = NULL,
            dedup_order = NULL{nb_set}
        WHEN NOT MATCHED AND s.action = 'UPSERT' THEN INSERT (
            id, pl_name, src_schema, src_tbl, trg_schema, trg_tbl,
            is_active, priority, load_mode, align_path, dedup_order{nb_col}
        ) VALUES (
            COALESCE((SELECT MAX(id) FROM {cfg}), 0) + s.seq,
            'PL_VV_TRANS_SLV_TO_SLV_1H', s.src_schema, s.src_tbl, s.trg_schema, s.trg_tbl,
            1, 1, 'RECOMPUTE', CAST(NULL AS STRING), CAST(NULL AS STRING){nb_val}
        )
    """)
    return int(missing), int(retired)


PL = "PL_VV_TRANS_SLV_TO_SLV_1H"
SETUP_EXEC = str(uuid.uuid4())
ensure_lock_row(PL, current_notebook_name("NB_SETUP_POI_ENTITY"))
ok, who = acquire_flow_lock(PL, SETUP_EXEC, SETUP_EXEC, LOCK_TIMEOUT_DEFAULT_MIN)
if not ok:
    raise ConfigError(f"Luồng {PL} đang chạy ({who}); chạy setup sau")
status = "FAILED"
try:
    EDGES_INSERTED, EDGES_RETIRED = with_retry(_write_edges, "edges slv_poi_entity_non_enrich")
    status = "SUCCESS"
    print(f"edges_inserted={EDGES_INSERTED} edges_retired={EDGES_RETIRED}")
finally:
    release_flow_lock(PL, SETUP_EXEC, status)


In [ ]:
import json

summary = {
    "status": "SUCCESS",
    "inserted": INSERTED,
    "drift": int(DRIFT),
    "edges_inserted": EDGES_INSERTED,
    "edges_retired": EDGES_RETIRED,
}
print(json.dumps(summary))
notebookutils.notebook.exit(json.dumps(summary))
